# Audiobook Summarization — Setup & Run Guide

## Pre-requisites (run locally once before opening Colab)

Run these scripts from the project root to upload data to GCS.  
They are resumable — re-run any time to pick up new books.

```bash
# 1. Upload raw book text/HTML for each book
python src/AudioBooks/BookSummary/book_contents_upload.py --workers 8 --chunk-size 200 --progress-every 50

# 2. Upload book metadata (title, author, reference summary) + Gutenberg→book-id map
python src/AudioBooks/BookSummary/book_desc_upload.py --workers 8 --chunk-size 200 --progress-every 50

# Check upload status at any time
python src/AudioBooks/BookSummary/book_contents_upload.py --status
python src/AudioBooks/BookSummary/book_desc_upload.py --status
```

GCS layout after upload:
```
gs://<bucket>/book-contents/<bookid>/clean_content.{txt,html}
gs://<bucket>/book-desc/<bookid>.json
gs://<bucket>/book-desc/gutenberg-id-map.json
```

---

## Colab Secrets (Colab → left sidebar → key icon)

| Secret key | Value |
|---|---|
| `GH_TOKEN` | GitHub personal access token (repo read access) |
| `HF_TOKEN` | Hugging Face token (for gated models) |

---

## File to upload to `/content` before running

Upload `credentials.json` (GCP service account key) to `/content/credentials.json`.  
Or place it in Google Drive at `MyDrive/AudioBooks/credentials.json` — the notebook searches both locations.

---

## Run order

1. **Install deps** — pip install cells + restart runtime
2. **Verify GPU** — confirm GPU is available via `nvidia-smi`
3. **Mount Drive + clone repo** — mounts `/content/drive` and clones the repo to `/content/AudioBooks`
4. **Config** — sets `MODEL_ID` and `GCS_BUCKET`
5. **Setup** — loads GCS book list, connects to HF Inference Endpoint, sets up generation + embedding + NLI models (run once per session)
6. **Batch loop** — summarizes books; set `MAX_BOOKS` for a smoke test, `None` for a full run.  
   Results append to `MyDrive/summary_results.jsonl` and the cell is safe to re-run (already-processed books are skipped).  
   Per-book checkpoints are saved to `MyDrive/summary_results_checkpoints/` so interrupted runs resume mid-book.

In [ ]:

%pip -q install -U gspread psutil transformers sentencepiece huggingface_hub google-cloud-storage python-dotenv


In [ ]:
gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)
if gpu_info.find('failed') >= 0:
  print('Not connected to a GPU')
else:
  print(gpu_info)

In [ ]:
import psutil

ram_gb = psutil.virtual_memory().total / 1e9
print('Your runtime has {:.1f} gigabytes of available RAM\n'.format(ram_gb))

if ram_gb < 20:
  print('Not using a high-RAM runtime')
else:
  print('You are using a high-RAM runtime!')

In [ ]:
import torch

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Version: {torch.version.cuda}")

In [ ]:
!pip install --upgrade pip setuptools wheel


In [ ]:
print('flash-attn is optional. Endpoint generation does not load a local model, so PyTorch SDPA is sufficient.')

In [ ]:
# No manual wheel download is needed for the Hugging Face endpoint workflow.

In [ ]:
# Intentionally empty: avoid installing an arbitrary flash-attn wheel.

In [ ]:
try:
    import flash_attn
    print("flash-attn installed successfully!")
except ImportError:
    print("flash-attn is not yet installed or failed to install.")

In [ ]:
try:
    from google.colab import drive
except ModuleNotFoundError as e:
    raise RuntimeError(
        "This notebook is intended to run in Google Colab only. "
        "Open it in Colab and rerun the cells there."
    ) from e

In [ ]:
try:
    import bitsandbytes
    print('bitsandbytes installed successfully!')
except ImportError:
    print('bitsandbytes is optional and is not required for endpoint generation.')

In [ ]:
import os
from pathlib import Path
def _credentials_path() -> Path:
    candidates = [
        Path(os.environ['GOOGLE_APPLICATION_CREDENTIALS']) if os.environ.get('GOOGLE_APPLICATION_CREDENTIALS') else None,
        Path('../credentials.json'),
        Path('/content/credentials.json'),
        Path('/content/drive/MyDrive/AudioBooks/credentials.json'),
    ]
    for candidate in candidates:
        if candidate is not None and candidate.is_file():
            return candidate
    raise FileNotFoundError(
        'Could not find credentials.json. Place it next to the notebook, set GOOGLE_APPLICATION_CREDENTIALS, or mount Drive in Colab.'
    )

from google.colab import drive
drive.mount('/content/drive', force_remount=False)
CREDENTIALS_PATH = _credentials_path()
os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = str(CREDENTIALS_PATH)

import gspread
gc = gspread.service_account(filename=str(CREDENTIALS_PATH))

In [ ]:
import os
import base64
import subprocess
from pathlib import Path
from google.colab import drive, userdata

# Read the repository token only from Colab Secrets. Never store it in the notebook.
try:
    GH_TOKEN = userdata.get('GH_TOKEN')
except Exception:
    GH_TOKEN = None
if not GH_TOKEN:
    raise RuntimeError('Add GH_TOKEN to Colab Secrets with read access to the repository.')

drive.mount('/content/drive', force_remount=False)

REPO_DIR = Path("/content/AudioBooks")

# Return to /content before cleanup
%cd /content

auth = base64.b64encode(f'x-access-token:{GH_TOKEN}'.encode()).decode()
git_env = os.environ.copy()
git_env.update({
    'GIT_CONFIG_COUNT': '1',
    'GIT_CONFIG_KEY_0': 'http.https://github.com/.extraheader',
    'GIT_CONFIG_VALUE_0': f'AUTHORIZATION: basic {auth}',
})
repo_url = 'https://github.com/helloani25/SystemDesign.git'
if REPO_DIR.exists():
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True, env=git_env)
else:
    print(f"Cloning into {REPO_DIR}")
    subprocess.run(['git', 'clone', repo_url, str(REPO_DIR)], check=True, env=git_env)
del auth, git_env, GH_TOKEN

%cd {REPO_DIR}
print(f"Success! REPO_DIR: {REPO_DIR}")

In [ ]:
import sys
SRC_ROOT = REPO_DIR / 'src'
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

In [ ]:
MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"
import os

GCS_BUCKET = os.environ.get("GCS_BUCKET", "gutenberg-books")

In [ ]:

import importlib
import json
import os
import time
import torch
from concurrent.futures import ThreadPoolExecutor
from google.colab import userdata
from huggingface_hub import InferenceClient, create_inference_endpoint, get_inference_endpoint
from transformers import AutoTokenizer

import AudioBooks.BookSummary.summarizer as ai_summarizer
ai_summarizer = importlib.reload(ai_summarizer)

_make_gcs_client       = ai_summarizer._make_gcs_client
_load_book_record      = ai_summarizer._load_book_record
summarize_book         = ai_summarizer.summarize_book
_semantic_similarity   = ai_summarizer._semantic_similarity
_lexical_similarity    = ai_summarizer._lexical_similarity
_max_nli_contradiction = ai_summarizer._max_nli_contradiction

HF_TOKEN = userdata.get('HF_TOKEN')
if not HF_TOKEN:
    raise RuntimeError('Add HF_TOKEN to Colab Secrets. It must be allowed to manage Inference Endpoints.')

gcs_client = _make_gcs_client(os.environ.get('GOOGLE_APPLICATION_CREDENTIALS'))

# Load full book ID list from GCS id map.
id_map_blob = gcs_client.bucket(GCS_BUCKET).blob('book-desc/gutenberg-id-map.json')
id_map: dict[str, int] = json.loads(id_map_blob.download_as_text(encoding='utf-8'))
all_book_ids = sorted(set(id_map.values()))
print(f"Found {len(all_book_ids)} books in GCS bucket '{GCS_BUCKET}'")

# Hugging Face Inference Endpoint configuration.
# Use a small repository such as 'gpt2' for endpoint smoke tests. For Qwen 7B,
# use a GPU instance and confirm your HF account has endpoint billing enabled.
ENDPOINT_NAME = 'audiobook-summary-qwen25-7b'
ENDPOINT_REPOSITORY = MODEL_ID
ENDPOINT_FRAMEWORK = 'pytorch'
ENDPOINT_TASK = 'text-generation'
ENDPOINT_ACCELERATOR = 'gpu'
ENDPOINT_VENDOR = 'aws'
ENDPOINT_REGION = 'us-east-1'
ENDPOINT_INSTANCE_SIZE = 'x1'
ENDPOINT_INSTANCE_TYPE = 'nvidia-a10g'
ENDPOINT_TYPE = 'authenticated'
CREATE_ENDPOINT_IF_MISSING = True


def load_or_create_endpoint():
    try:
        endpoint = get_inference_endpoint(ENDPOINT_NAME, token=HF_TOKEN)
        print(f"Using existing endpoint: {ENDPOINT_NAME}")
    except Exception as exc:
        if not CREATE_ENDPOINT_IF_MISSING:
            raise
        print(f"Creating endpoint {ENDPOINT_NAME}: {exc}")
        endpoint = create_inference_endpoint(
            ENDPOINT_NAME,
            repository=ENDPOINT_REPOSITORY,
            framework=ENDPOINT_FRAMEWORK,
            task=ENDPOINT_TASK,
            accelerator=ENDPOINT_ACCELERATOR,
            vendor=ENDPOINT_VENDOR,
            region=ENDPOINT_REGION,
            instance_size=ENDPOINT_INSTANCE_SIZE,
            instance_type=ENDPOINT_INSTANCE_TYPE,
            type=ENDPOINT_TYPE,
            token=HF_TOKEN,
        )

    endpoint.wait()
    print(f"Endpoint URL: {endpoint.url}")
    return endpoint


endpoint = load_or_create_endpoint()
hf_client = InferenceClient(model=endpoint.url, token=HF_TOKEN)

# Tokenizer runs locally for prompt/chunk sizing. Model generation runs on the HF endpoint.
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=HF_TOKEN)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token or tokenizer.unk_token or '[PAD]'

# Chunks shorter than this token count are skipped (usually just headings with no body).
MIN_CHUNK_TOKENS = 20


def _truncate_at_prompt_marker(text: str) -> str:
    idx = text.find("###")
    return text[:idx].strip() if idx != -1 else text.strip()


def _call_hf_endpoint(prompt: str, *, max_new_tokens: int) -> str:
    print(
        f"Calling HF endpoint {ENDPOINT_NAME} for {len(prompt):,} prompt chars, "
        f"max_new_tokens={max_new_tokens}",
        flush=True,
    )
    for attempt in range(3):
        try:
            result = hf_client.text_generation(
                prompt,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                repetition_penalty=1.05,
                return_full_text=False,
            )
            result = _truncate_at_prompt_marker(result)
            print(f"RAW MODEL OUTPUT: {result!r}", flush=True)
            return result
        except Exception as exc:
            if attempt == 2:
                raise
            wait_seconds = 5 * (attempt + 1)
            print(f"Endpoint generation failed ({exc}); retrying in {wait_seconds}s", flush=True)
            time.sleep(wait_seconds)


def _remote_generate_text(model, tokenizer, prompt: str, *, device, max_input_tokens: int, max_new_tokens: int) -> str:
    input_ids = tokenizer(prompt, add_special_tokens=False, truncation=True, max_length=max_input_tokens).input_ids
    truncated_prompt = tokenizer.decode(input_ids, skip_special_tokens=True)
    generated = _call_hf_endpoint(truncated_prompt, max_new_tokens=max_new_tokens)
    return ai_summarizer._strip_prompt(generated)


def _remote_generate_batch(model, tokenizer, prompts: list[str], *, device, max_input_tokens: int, max_new_tokens: int) -> list[str]:
    # Filter out trivial prompts (heading-only chunks) before sending to the endpoint.
    results = [''] * len(prompts)
    work = [
        (i, p) for i, p in enumerate(prompts)
        if len(tokenizer(p, add_special_tokens=False).input_ids) >= MIN_CHUNK_TOKENS
    ]
    skipped = len(prompts) - len(work)
    if skipped:
        print(f"  skipping {skipped} trivial chunk(s) with < {MIN_CHUNK_TOKENS} tokens", flush=True)
    if not work:
        return results
    with ThreadPoolExecutor(max_workers=BATCH_SIZE) as executor:
        futures = {
            executor.submit(
                _remote_generate_text,
                model, tokenizer, prompt,
                device=device,
                max_input_tokens=max_input_tokens,
                max_new_tokens=max_new_tokens,
            ): idx
            for idx, prompt in work
        }
        for future, idx in futures.items():
            results[idx] = future.result()
    return results


ai_summarizer._generate_text = _remote_generate_text
ai_summarizer._generate_batch = _remote_generate_batch

model = hf_client
print('HF cloud generation is configured. No local generation model was loaded.')


In [ ]:

from pathlib import Path

# Configuration
# Set BOOK_ID_TO_SUMMARIZE to one known internal book_id for a single-book run.
# Leave it as None to resume through the first MAX_BOOKS unprocessed books.
BOOK_ID_TO_SUMMARIZE = None
MAX_BOOKS = 20
RUN_VALIDATION = False  # Keep False to avoid loading embedding/NLI models locally.
SEMANTIC_THRESHOLD = 0.60
LEXICAL_FLOOR = 0.35
NLI_THRESHOLD = 0.50
CHUNK_TOKENS = 4096
CHUNK_OVERLAP = 400
REDUCE_INPUT_TOKENS = 8192
MAX_NEW_TOKENS = 512
REDUCE_MAX_NEW_TOKENS = 768
PROFILE_MAX_NEW_TOKENS = 1024
MAX_CHAPTERS = None
MAX_CHUNKS_PER_CHAPTER = None
BATCH_SIZE = 4
OUTPUT_PATH = Path('/content/drive/MyDrive/summary_results.jsonl')
CHECKPOINT_DIR = Path('/content/drive/MyDrive/summary_results_checkpoints')

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

embedding_tokenizer = embedding_model = None
nli_tokenizer = nli_model = None
if RUN_VALIDATION:
    embedding_tokenizer, embedding_model = ai_summarizer._load_embedding_model(
        ai_summarizer.DEFAULT_EMBEDDING_MODEL_ID, HF_TOKEN
    )
    nli_tokenizer, nli_model = ai_summarizer._load_nli_model(
        ai_summarizer.DEFAULT_NLI_MODEL_ID, HF_TOKEN
    )

processed_ids: set[int] = set()
if OUTPUT_PATH.exists() and BOOK_ID_TO_SUMMARIZE is None:
    for line in OUTPUT_PATH.read_text(encoding='utf-8').splitlines():
        try:
            processed_ids.add(json.loads(line)['book_id'])
        except Exception:
            pass
print(f'Already processed: {len(processed_ids)} books')

if BOOK_ID_TO_SUMMARIZE is not None:
    book_ids_to_run = [BOOK_ID_TO_SUMMARIZE]
else:
    book_ids_to_run = [bid for bid in all_book_ids if bid not in processed_ids]
    if MAX_BOOKS is not None:
        book_ids_to_run = book_ids_to_run[:MAX_BOOKS]
print(f'Running: {len(book_ids_to_run)} books via HF endpoint {ENDPOINT_NAME}')


def summarize_one_book_via_hf_endpoint(book_id: int) -> dict | None:
    print(f"\nbook_id={book_id}", flush=True)
    try:
        book, book_desc_summary = _load_book_record(gcs_client, GCS_BUCKET, book_id)
    except FileNotFoundError as e:
        print(f'  SKIP: {e}', flush=True)
        return None

    mode = 'verify' if book_desc_summary and RUN_VALIDATION else 'generate'
    print(f'  mode={mode} title={book.title!r} category={book.category or "(unknown)"!r}', flush=True)
    print(f'  subjects={book.subjects}', flush=True)
    print(f'  generation_endpoint={endpoint.url}', flush=True)

    checkpoint_path = str(CHECKPOINT_DIR / f'checkpoint_{book_id}.jsonl')

    result = summarize_book(
        model,
        tokenizer,
        book,
        device=None,
        chunk_tokens=CHUNK_TOKENS,
        chunk_overlap=CHUNK_OVERLAP,
        reduce_input_tokens=REDUCE_INPUT_TOKENS,
        max_new_tokens=MAX_NEW_TOKENS,
        reduce_max_new_tokens=REDUCE_MAX_NEW_TOKENS,
        batch_size=BATCH_SIZE,
        max_chapters=MAX_CHAPTERS,
        max_chunks_per_chapter=MAX_CHUNKS_PER_CHAPTER,
        story_so_far_tokens=REDUCE_MAX_NEW_TOKENS,
        checkpoint_path=checkpoint_path,
        profile_max_new_tokens=PROFILE_MAX_NEW_TOKENS,
    )

    result['mode'] = mode
    result['book_desc_summary'] = book_desc_summary
    result['semantic_score'] = None
    result['lexical_score'] = None
    result['nli_contradiction_score'] = None
    result['similarity_pass'] = None

    # Print the final story-so-far (last chapter's running summary).
    if result.get('chapters'):
        final_story_so_far = result['chapters'][-1].get('story_so_far', '')
        if final_story_so_far:
            print('\nSTORY SO FAR (end of book)')
            print(final_story_so_far)

    print('\nFINAL SUMMARY')
    print(result['final_summary'])

    profiles = result.get('character_profiles', '')
    if profiles:
        print(f"\nCHARACTER PROFILES  [category={result.get('category', '?')!r}]")
        print(profiles)

    if RUN_VALIDATION and book_desc_summary:
        semantic = _semantic_similarity(
            embedding_tokenizer,
            embedding_model,
            result['final_summary'],
            book_desc_summary,
        )
        lexical = _lexical_similarity(result['final_summary'], book_desc_summary)
        contradiction = _max_nli_contradiction(
            nli_tokenizer,
            nli_model,
            result['final_summary'],
            book_desc_summary,
        )
        sem_pass = semantic >= SEMANTIC_THRESHOLD
        lex_pass = lexical >= LEXICAL_FLOOR
        nli_pass = contradiction < NLI_THRESHOLD
        sim_pass = sem_pass and (lex_pass or nli_pass)
        result.update({
            'semantic_score': semantic,
            'lexical_score': lexical,
            'nli_contradiction_score': contradiction,
            'similarity_pass': sim_pass,
        })
        print(
            f'  semantic={semantic:.3f} lexical={lexical:.3f}'
            f' nli_contradiction={contradiction:.3f} pass={sim_pass}',
            flush=True,
        )

    return result


for idx, book_id in enumerate(book_ids_to_run, 1):
    print(f"\n[{idx}/{len(book_ids_to_run)}]", flush=True)
    try:
        result = summarize_one_book_via_hf_endpoint(book_id)
    except Exception as e:
        print(f'  ERROR summarizing: {e}', flush=True)
        continue

    if result is None:
        continue

    with open(OUTPUT_PATH, 'a', encoding='utf-8') as f:
        f.write(json.dumps(result, ensure_ascii=False) + '\n')
    print(f'  saved -> {OUTPUT_PATH}', flush=True)

print('\nDone.')
